# ViPIBench Confirmatory Execution Controller

## Purpose

This internal notebook executes the frozen confirmatory protocol for the graduation internship project. It validates the isolated interpreter, verifies the exact dependency lock and hash-bound launch package, measures the observed accelerator, and coordinates checkpointed execution of the declared experiment stages.

## Operational controls

- Confirmatory mode requires explicit, run-scoped authorization for the uploaded bundle and allocated accelerator compute.
- Every public stage is restricted to the declared NVIDIA A100 80 GB runtime profile and does not silently fall back to CPU or a smaller accelerator. The same runtime must remain active through `analysis` and `finalize`; host-side statistical and file-system operations inside that runtime are not claimed to be CUDA kernels.
- Checkpoint resume is accepted only when stage metadata and output hashes match the current launch package.
- Model selection and autonomous stopping decisions use development evidence only; the final holdout cannot control reruns or capacity decisions.

## Evidence and authority boundary

This notebook does not mount external storage, upload data, publish artifacts, or authorize release. Those responsibilities belong to the outer launcher and its durable snapshot controller. A successful run produces evidence eligible for the locked analysis protocol; it does not by itself establish positive research hypotheses or production readiness.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

EXPECTED_RUNTIME_PYTHON = os.environ.get("VIPIBENCH_RUNTIME_PYTHON", "").strip()
OBSERVED_RUNTIME_PYTHON = str(Path(sys.executable).absolute())
if (
    not EXPECTED_RUNTIME_PYTHON
    or str(Path(EXPECTED_RUNTIME_PYTHON).absolute()) != OBSERVED_RUNTIME_PYTHON
):
    raise RuntimeError(
        {
            "error": "notebook_kernel_interpreter_mismatch",
            "expected": EXPECTED_RUNTIME_PYTHON,
            "observed": OBSERVED_RUNTIME_PYTHON,
        }
    )

MODE = os.environ.get("VIPIBENCH_MODE", "smoke").strip().lower()
SELECTED_STAGE = os.environ.get("VIPIBENCH_STAGE", "preflight").strip().lower()
DEPENDENCY_PROFILE = os.environ.get("VIPIBENCH_DEPENDENCY_PROFILE", "accelerator").strip().lower()
SESSION_ID = os.environ.get("VIPIBENCH_SESSION_ID", "").strip()
PROJECT_ROOT = Path(os.environ.get("VIPIBENCH_PROJECT_ROOT", ".")).resolve()
dataset_default = PROJECT_ROOT / "data" / "processed" / "vipibench_exec.jsonl"
split_default = PROJECT_ROOT / "data" / "splits" / "confirmatory_final"
DATASET_PATH = Path(os.environ.get("VIPIBENCH_DATASET_PATH", dataset_default)).resolve()
SPLIT_DIR = Path(os.environ.get("VIPIBENCH_SPLIT_DIR", split_default)).resolve()
OUTPUT_ROOT_SETTING = os.environ.get("VIPIBENCH_OUTPUT_ROOT")
OUTPUT_ROOT = Path(OUTPUT_ROOT_SETTING or PROJECT_ROOT / "outputs" / "run").resolve()
SESSION_EVIDENCE_ROOT = Path(
    os.environ.get("VIPIBENCH_SESSION_EVIDENCE_ROOT", PROJECT_ROOT / "outputs")
).resolve()
default_cache_root = (
    Path("/content/vipibench-model-cache")
    if Path("/content").is_dir()
    else PROJECT_ROOT / "build" / "vipibench-model-cache"
)
EPHEMERAL_MODEL_CACHE_ROOT = Path(
    os.environ.get("VIPIBENCH_EPHEMERAL_MODEL_CACHE_ROOT", default_cache_root)
).resolve()
os.environ["HF_HOME"] = str(EPHEMERAL_MODEL_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(EPHEMERAL_MODEL_CACHE_ROOT / "hub")
if MODE not in {"smoke", "confirmatory"}:
    raise ValueError("VIPIBENCH_MODE must be smoke or confirmatory")
if MODE == "confirmatory" and not SESSION_ID:
    raise ValueError("VIPIBENCH_SESSION_ID is required in confirmatory mode")
if DEPENDENCY_PROFILE != "accelerator":
    raise PermissionError("Every public stage requires the accelerator dependency profile")
SESSION_RUNTIME_ROOT = SESSION_EVIDENCE_ROOT / "runtime_sessions" / (SESSION_ID or "smoke")
print(
    json.dumps(
        {
            "mode": MODE,
            "public_stage": SELECTED_STAGE,
            "dependency_profile": DEPENDENCY_PROFILE,
            "project_root": str(PROJECT_ROOT),
            "output_root": str(OUTPUT_ROOT),
        },
        ensure_ascii=True,
    )
)

In [ ]:
if MODE == "confirmatory":
    CACHE_MARKER_TEXT = "vipibench-owned-ephemeral-model-cache"
    cache_marker = EPHEMERAL_MODEL_CACHE_ROOT / ".vipibench-owned-cache"
    if (
        EPHEMERAL_MODEL_CACHE_ROOT == PROJECT_ROOT
        or EPHEMERAL_MODEL_CACHE_ROOT in PROJECT_ROOT.parents
    ):
        raise ValueError("Ephemeral model cache cannot equal or contain the project root")
    if (
        EPHEMERAL_MODEL_CACHE_ROOT == OUTPUT_ROOT
        or EPHEMERAL_MODEL_CACHE_ROOT in OUTPUT_ROOT.parents
        or OUTPUT_ROOT in EPHEMERAL_MODEL_CACHE_ROOT.parents
    ):
        raise ValueError("Ephemeral model cache and durable output root must not overlap")
    if (
        EPHEMERAL_MODEL_CACHE_ROOT.exists()
        and any(EPHEMERAL_MODEL_CACHE_ROOT.iterdir())
        and not cache_marker.is_file()
    ):
        raise FileExistsError("Refusing to use or clean a non-empty unmarked model cache")
    EPHEMERAL_MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    if cache_marker.is_file() and cache_marker.read_text(encoding="utf-8") != CACHE_MARKER_TEXT:
        raise ValueError("Ephemeral model-cache ownership marker mismatch")
    cache_marker.write_text(CACHE_MARKER_TEXT, encoding="utf-8")

In [ ]:
LOCK_PATH = PROJECT_ROOT / "requirements-experiment.lock"
PREPARE_SCRIPT = PROJECT_ROOT / "scripts" / "prepare_colab.py"
if not LOCK_PATH.is_file():
    raise FileNotFoundError(f"Missing exact experiment lock: {LOCK_PATH}")
if not PREPARE_SCRIPT.is_file():
    raise FileNotFoundError(f"Missing Colab environment probe: {PREPARE_SCRIPT}")
RUNTIME_PROBE_COMMAND = [
    sys.executable,
    str(PREPARE_SCRIPT),
    "--project-root",
    str(PROJECT_ROOT),
    "--probe-only",
    "--dependency-profile",
    DEPENDENCY_PROFILE,
]
PROJECT_SOURCE_ROOT = (PROJECT_ROOT / "src").resolve()
if not PROJECT_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"Missing staged project source: {PROJECT_SOURCE_ROOT}")
subprocess.check_call(RUNTIME_PROBE_COMMAND)
if str(PROJECT_SOURCE_ROOT) not in sys.path:
    raise RuntimeError(
        {"error": "project_source_not_on_pythonpath", "path": str(PROJECT_SOURCE_ROOT)}
    )
import vipibench  # noqa: E402, I001
resolved_package = Path(vipibench.__file__).resolve()
if PROJECT_SOURCE_ROOT not in resolved_package.parents:
    raise RuntimeError(
        {
            "error": "project_source_import_mismatch",
            "expected_source_root": str(PROJECT_SOURCE_ROOT),
            "resolved_package": str(resolved_package),
        }
    )

In [ ]:
SESSION_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
ENVIRONMENT_COMPATIBILITY_PATH = SESSION_RUNTIME_ROOT / "environment_compatibility.json"
ENVIRONMENT_COMMAND = [
    sys.executable,
    "-m",
    "vipibench.cli",
    "verify-environment-compatibility",
    "--project-root",
    str(PROJECT_ROOT),
    "--output",
    str(ENVIRONMENT_COMPATIBILITY_PATH),
]
subprocess.check_call(ENVIRONMENT_COMMAND, cwd=PROJECT_ROOT)

In [ ]:
if DEPENDENCY_PROFILE == "accelerator":
    PREFLIGHT_PATH = SESSION_RUNTIME_ROOT / "prelaunch_readiness.json"
    PREFLIGHT_COMMAND = [
        sys.executable,
        "-m",
        "vipibench.cli",
        "preflight",
        "--project-root",
        str(PROJECT_ROOT),
        "--verify-hash",
        "--runtime-environment-compatibility",
        str(ENVIRONMENT_COMPATIBILITY_PATH),
        "--active-isolated-runtime",
        "--output",
        str(PREFLIGHT_PATH),
    ]
    PREFLIGHT_PROCESS = subprocess.run(
        PREFLIGHT_COMMAND,
        cwd=PROJECT_ROOT,
        check=False,
        capture_output=True,
        text=True,
    )
PREFLIGHT = None
preflight_receipt_error = None
if PREFLIGHT_PATH.is_file():
    try:
        PREFLIGHT = json.loads(PREFLIGHT_PATH.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        preflight_receipt_error = f"{type(exc).__name__}: {exc}"
if PREFLIGHT_PROCESS.returncode != 0:
    failed_check_evidence = {}
    if isinstance(PREFLIGHT, dict):
        for check in PREFLIGHT.get("checks", []):
            if isinstance(check, dict) and check.get("status") != "PASS":
                evidence = check.get("evidence")
                if check.get("name") == "confirmatory_readiness" and isinstance(evidence, dict):
                    nested = {
                        str(item.get("name")): item.get("evidence")
                        for item in evidence.get("checks", [])
                        if isinstance(item, dict) and item.get("status") != "PASS"
                    }
                    failed_check_evidence[str(check.get("name"))] = {
                        "failed_checks": evidence.get("failed_checks"),
                        "failed_check_evidence": nested,
                    }
                else:
                    failed_check_evidence[str(check.get("name"))] = evidence
    diagnostic = {
        "stage": "preflight",
        "returncode": PREFLIGHT_PROCESS.returncode,
        "receipt_path": str(PREFLIGHT_PATH),
        "receipt_status": PREFLIGHT.get("status") if isinstance(PREFLIGHT, dict) else None,
        "milestone": PREFLIGHT.get("milestone") if isinstance(PREFLIGHT, dict) else None,
        "failed_checks": PREFLIGHT.get("failed_checks") if isinstance(PREFLIGHT, dict) else None,
        "failed_check_evidence": failed_check_evidence,
        "receipt_error": preflight_receipt_error,
        "cli_output_tail": (PREFLIGHT_PROCESS.stdout or "")[-2000:],
    }
    raise RuntimeError({"preflight_failed": diagnostic})
if not isinstance(PREFLIGHT, dict):
    raise RuntimeError({"preflight_receipt_missing_or_invalid": str(PREFLIGHT_PATH)})
if (
    PREFLIGHT.get("status") != "PASS"
    or PREFLIGHT.get("milestone") != "READY_FOR_CONFIRMATORY_LAUNCH"
):
    raise RuntimeError(
        {
            "preflight_status": PREFLIGHT.get("status"),
            "failed_checks": PREFLIGHT.get("failed_checks"),
        }
    )

In [ ]:
if MODE == "smoke":
    required_paths = [PROJECT_ROOT / "pyproject.toml", DATASET_PATH, SPLIT_DIR / "manifest.json"]
    missing = [str(path) for path in required_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError({"missing_smoke_contract_paths": missing})
    print({"status": "PASS", "mode": MODE, "checked_paths": len(required_paths)})

In [ ]:
if MODE == "confirmatory":
    if os.environ.get("VIPIBENCH_CONFIRMATORY_RUN_APPROVED") != "YES":
        raise PermissionError("VIPIBENCH_CONFIRMATORY_RUN_APPROVED=YES is required")
    if os.environ.get("VIPIBENCH_DURABLE_OUTPUT_CONFIRMED") != "YES":
        raise PermissionError("VIPIBENCH_DURABLE_OUTPUT_CONFIRMED=YES is required")
    if os.environ.get("VIPIBENCH_UPLOAD_AUTHORIZED") != "YES":
        raise PermissionError("VIPIBENCH_UPLOAD_AUTHORIZED=YES is required")
    if os.environ.get("VIPIBENCH_PAID_COMPUTE_AUTHORIZED") != "YES":
        raise PermissionError("VIPIBENCH_PAID_COMPUTE_AUTHORIZED=YES is required")
    if not OUTPUT_ROOT_SETTING:
        raise ValueError("VIPIBENCH_OUTPUT_ROOT must explicitly name the durable run directory")
    from vipibench.dataio import sha256_file, write_json

    authorization_path = Path(os.environ.get("VIPIBENCH_LAUNCH_AUTHORIZATION_PATH", "")).resolve()
    authorization_sha256 = os.environ.get("VIPIBENCH_LAUNCH_AUTHORIZATION_SHA256", "")
    if not authorization_path.is_file() or sha256_file(authorization_path) != authorization_sha256:
        raise PermissionError("launch authorization record is missing or hash-mismatched")
    authorization = json.loads(authorization_path.read_text(encoding="utf-8"))
    scopes = authorization.get("scopes", {})
    budget = authorization.get("budget", {})
    if (
        authorization.get("status") != "AUTHORIZED"
        or scopes.get("drive_upload") is not True
        or scopes.get("paid_compute") is not True
    ):
        raise PermissionError("launch authorization scopes are incomplete")
    if scopes.get("public_release") is not False or scopes.get("publication") is not False:
        raise PermissionError("launch authorization must not imply release or publication")
    autonomous_policy_path = Path(
        os.environ.get("VIPIBENCH_AUTONOMOUS_EXECUTION_POLICY_PATH", "")
    ).resolve()
    autonomous_policy_sha256 = os.environ.get("VIPIBENCH_AUTONOMOUS_EXECUTION_POLICY_SHA256", "")
    if (
        not autonomous_policy_path.is_file()
        or sha256_file(autonomous_policy_path) != autonomous_policy_sha256
    ):
        raise PermissionError("autonomous execution policy is missing or hash-mismatched")
    from vipibench.autonomous_runtime import load_autonomous_execution_policy

    autonomous_policy = load_autonomous_execution_policy(autonomous_policy_path)
    if (
        budget.get("mode") != "autonomous"
        or budget.get("policy_sha256") != autonomous_policy_sha256
    ):
        raise PermissionError("launch authorization is not bound to the autonomous policy")
    if (
        float(budget.get("hard_ceiling_hours_per_session", 0))
        != autonomous_policy.hard_ceiling_hours
    ):
        raise PermissionError("autonomous session ceiling binding mismatch")
    if (
        budget.get("final_holdout_feedback_allowed") is not False
        or autonomous_policy.final_holdout_feedback_allowed is not False
    ):
        raise PermissionError("final holdout cannot control autonomous execution")
    resume_existing = os.environ.get("VIPIBENCH_RESUME_EXISTING_RUN") == "YES"
    from vipibench.launch_contract import validate_new_run_output_root

    validate_new_run_output_root(
        OUTPUT_ROOT, SESSION_EVIDENCE_ROOT, resume_existing=resume_existing
    )
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    from vipibench.cache_contract import reset_ephemeral_model_cache
    from vipibench.checkpoint import StageLedger
    from vipibench.stage_orchestration import (
        build_stage_status,
        checkpoint_metadata,
        complete_stage_group,
        load_stage_plan,
        require_stage_prerequisite,
        stage_enabled,
        validate_stage_selection,
    )

    STAGE_PLAN_PATH = Path(os.environ.get("VIPIBENCH_STAGE_PLAN_PATH", "")).resolve()
    expected_stage_plan_sha256 = os.environ.get("VIPIBENCH_STAGE_PLAN_SHA256", "")
    if (
        not STAGE_PLAN_PATH.is_file()
        or sha256_file(STAGE_PLAN_PATH) != expected_stage_plan_sha256
    ):
        raise PermissionError("confirmatory stage plan is missing or hash-mismatched")
    STAGE_PLAN = load_stage_plan(STAGE_PLAN_PATH)
    SELECTED_STAGE = validate_stage_selection(STAGE_PLAN, SELECTED_STAGE)
    protocol_amendment = os.environ.get("VIPIBENCH_PROTOCOL_AMENDMENT", "")
    durable_lineage = os.environ.get("VIPIBENCH_DURABLE_LINEAGE", "")
    if (
        authorization.get("protocol_amendment") != protocol_amendment
        or authorization.get("durable_lineage") != durable_lineage
        or authorization.get("stage_plan_sha256") != expected_stage_plan_sha256
    ):
        raise PermissionError("launch authorization stage binding mismatch")
    RUN_BINDING = {
        "runtime_source_fingerprint": PREFLIGHT["launch_hashes"]["runtime_source_fingerprint"],
        "launch_hashes": PREFLIGHT["launch_hashes"],
        "launch_authorization_sha256": authorization_sha256,
        "stage_plan_sha256": expected_stage_plan_sha256,
        "protocol_amendment": protocol_amendment,
        "durable_lineage": durable_lineage,
    }
    ORCHESTRATION_LEDGER = StageLedger(
        OUTPUT_ROOT / "orchestration_ledger", artifact_root=OUTPUT_ROOT
    )
    STAGE_GROUP_LEDGER = StageLedger(
        OUTPUT_ROOT / "stage_group_ledger", artifact_root=OUTPUT_ROOT
    )
    MEMBER_TO_PUBLIC_STAGE = {
        str(member): str(stage["id"])
        for stage in STAGE_PLAN["stages"]
        for member in stage["member_stage_ids"]
    }
    CLI = [sys.executable, "-m", "vipibench.cli"]

    def stage_reader_label(stage_id):
        labels = {
            "core-target-trajectories": "sinh quỹ đạo phản hồi của mô hình trên tập kiểm tra",
            "attack-target-trajectories": "sinh quỹ đạo phản hồi của mô hình trên tập tấn công",
            "analyze-rq2-diagnostics": "phân tích khả năng khái quát của bộ phát hiện",
            "analyze-h3": "phân tích hiệu quả của cơ chế bảo vệ kết hợp",
            "analyze-adaptive-search": "phân tích tấn công thích nghi",
        }
        return labels.get(stage_id, "bước xử lý thí nghiệm")

    def reader_safe_failure_line(line):
        safe = line
        for prefix in ("RQ", "H"):
            for number in range(1, 10):
                safe = safe.replace(f"{prefix}{number}", "kết luận nghiên cứu")
                safe = safe.replace(f"{prefix.lower()}{number}", "phân_tích_nghiên_cứu")
        replacements = {
            "INCONCLUSIVE_FORMAT_FALLBACK": "CHƯA THỂ KẾT LUẬN DO ĐẦU RA SAI CẤU TRÚC",
            "INCONCLUSIVE_PARSE_FAILURE": "CHƯA THỂ KẾT LUẬN DO KHÔNG ĐỌC ĐƯỢC ĐẦU RA",
            "DEFERRED_FROZEN_EVALUATION": "CHỜ ĐÁNH GIÁ THEO QUY TRÌNH ĐÃ KHÓA",
        }
        for machine_value, reader_value in replacements.items():
            safe = safe.replace(machine_value, reader_value)
        return safe

    def stage_failure_receipts(stage_id):
        if stage_id == "encoder-matrix":
            encoder_root = OUTPUT_ROOT / "mdeberta"
            candidates = [
                encoder_root / "capacity_plan.json",
                encoder_root / "dataloader_worker_plan.json",
            ]
            candidates.extend(sorted(encoder_root.glob("*/numerical_failure.json")))
        elif stage_id == "attack-candidate-generation":
            candidates = sorted((OUTPUT_ROOT / "attack_search_checkpoints").glob("*.failure.json"))
        elif stage_id in {"core-target-trajectories", "attack-target-trajectories"}:
            run_receipt = (
                "core_target_trajectories.run.json"
                if stage_id == "core-target-trajectories"
                else "attack_target_trajectories.run.json"
            )
            candidates = [OUTPUT_ROOT / run_receipt]
        else:
            return []
        summaries = []
        for path in candidates:
            if not path.is_file():
                continue
            try:
                payload = json.loads(path.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError) as exc:
                summaries.append({
                    "path": str(path),
                    "read_error": f"{type(exc).__name__}: {exc}",
                })
                continue
            summary = {
                "path": str(path),
                "status": payload.get("status"),
                "errors": payload.get("errors"),
            }
            for field in (
                "run_id",
                "error_type",
                "message",
                "resume_allowed",
                "measurements",
            ):
                if field in payload:
                    summary[field] = payload[field]
            selected = payload.get("selected")
            if isinstance(selected, dict):
                summary["selected"] = {
                    key: selected.get(key)
                    for key in ("candidate_id", "batch_size", "peak_reserved_gib")
                }
            canary = payload.get("numerics_canary")
            if isinstance(canary, dict):
                summary["numerics_canary"] = {
                    "status": canary.get("status"),
                    "candidate_id": canary.get("candidate_id"),
                    "attempts": [
                        {
                            key: attempt.get(key)
                            for key in ("status", "candidate_id", "error_type", "message")
                            if key in attempt
                        }
                        for attempt in canary.get("attempts", [])
                        if isinstance(attempt, dict)
                    ],
                }
            format_summary = payload.get("format_failure_summary")
            if isinstance(format_summary, dict):
                summary["format_failure_summary"] = {
                    key: format_summary.get(key)
                    for key in (
                        "status",
                        "format_fallback_count",
                        "parse_failure_count",
                        "parse_error_class_counts",
                        "truncated_response_count",
                        "response_token_ceiling",
                        "response_token_ceiling_reached_count",
                        "raw_response_included",
                    )
                }
            fail_fast = payload.get("target_format_fail_fast")
            if isinstance(fail_fast, dict):
                summary["target_format_fail_fast"] = {
                    key: fail_fast.get(key)
                    for key in (
                        "recorded_episode_count",
                        "total_episode_count",
                        "unprocessed_episode_count",
                        "additional_model_batches_after_trigger",
                    )
                }
            summaries.append(summary)
        return summaries

    def print_stage_failure_tail(stage_id, output_tail, related_receipts):
        for receipt in related_receipts:
            format_summary = receipt.get("format_failure_summary")
            if not isinstance(format_summary, dict):
                continue
            fail_fast = receipt.get("target_format_fail_fast", {})
            truncated_count = format_summary.get("truncated_response_count") or 0
            if truncated_count:
                print(
                    "Thí nghiệm đã dừng vì phản hồi của mô hình bị cắt ngang khi đạt giới hạn "
                    "số token đầu ra đã đăng ký, nên chưa quan sát được phần kết thúc của phản hồi.",
                    flush=True,
                )
                print(
                    "Số phản hồi bị cắt ngang: "
                    f"{truncated_count}; giới hạn token đầu ra hiện tại: "
                    f"{format_summary.get('response_token_ceiling')}.",
                    flush=True,
                )
                print(
                    "Phản hồi bị cắt ngang không được vá bằng dấu đóng, vì làm vậy sẽ chấp nhận "
                    "một quỹ đạo chưa hoàn chỉnh như một quan sát hợp lệ.",
                    flush=True,
                )
            else:
                print(
                    "Thí nghiệm đã dừng vì phản hồi của mô hình không đúng cấu trúc JSON "
                    "đã quy định.",
                    flush=True,
                )
            total_label = (
                "Tổng số phản hồi không dùng được"
                if truncated_count
                else "Số phản hồi sai cấu trúc"
            )
            print(
                f"{total_label}: "
                f"{format_summary.get('parse_failure_count', 0)}; "
                "số mẫu chưa xử lý: "
                f"{fail_fast.get('unprocessed_episode_count', 0)}.",
                flush=True,
            )
            print(
                "Lần chạy này chưa đủ điều kiện để rút ra kết luận nghiên cứu. "
                "Bằng chứng lỗi kỹ thuật đã được lưu để kiểm tra.",
                flush=True,
            )
            return
        print(f"Không hoàn tất {stage_reader_label(stage_id)}.", flush=True)
        for line in output_tail.splitlines()[-20:]:
            bounded = line if len(line) <= 1000 else "..." + line[-997:]
            print(reader_safe_failure_line(bounded), flush=True)

    def run_checkpointed_stage(stage_id, command, outputs):
        public_stage = MEMBER_TO_PUBLIC_STAGE.get(stage_id)
        if public_stage is None:
            raise ValueError(f"checkpointed stage is absent from the locked plan: {stage_id}")
        if not stage_enabled(SELECTED_STAGE, public_stage):
            print({"stage": stage_id, "action": "stage-selection-skip"})
            return
        require_stage_prerequisite(
            plan=STAGE_PLAN,
            stage_id=public_stage,
            stage_ledger=ORCHESTRATION_LEDGER,
            group_ledger=STAGE_GROUP_LEDGER,
            output_root=OUTPUT_ROOT,
            run_binding=RUN_BINDING,
        )
        metadata = checkpoint_metadata(command, RUN_BINDING)
        if ORCHESTRATION_LEDGER.verified_complete(stage_id, metadata):
            print({"stage": stage_id, "action": "resume-skip-verified-complete"})
            return
        process = subprocess.run(
            command,
            cwd=PROJECT_ROOT,
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            errors="replace",
        )
        if process.returncode:
            failure_path = OUTPUT_ROOT / "stage_failures" / f"{stage_id}.json"
            output_tail = (process.stdout or "")[-12000:]
            related_receipts = stage_failure_receipts(stage_id)
            diagnostic = {
                "schema_version": "1.0.0",
                "status": "FAIL",
                "stage": stage_id,
                "returncode": process.returncode,
                "command": [str(value) for value in command],
                "output_tail": output_tail,
                "related_receipts": related_receipts,
            }
            write_json(failure_path, diagnostic)
            print_stage_failure_tail(stage_id, output_tail, related_receipts)
            raise RuntimeError(
                f"Không hoàn tất {stage_reader_label(stage_id)}. "
                "Biên bản chẩn đoán kỹ thuật đã được lưu."
            )
        ORCHESTRATION_LEDGER.complete(stage_id, outputs, metadata)

    def complete_public_stage(stage_id, direct_outputs=None, *, always=False):
        if not always and not stage_enabled(SELECTED_STAGE, stage_id):
            return
        complete_stage_group(
            plan=STAGE_PLAN,
            stage_id=stage_id,
            stage_ledger=ORCHESTRATION_LEDGER,
            group_ledger=STAGE_GROUP_LEDGER,
            output_root=OUTPUT_ROOT,
            run_binding=RUN_BINDING,
            direct_outputs=direct_outputs or [],
        )
        status = build_stage_status(
            plan=STAGE_PLAN,
            stage_ledger=ORCHESTRATION_LEDGER,
            group_ledger=STAGE_GROUP_LEDGER,
            output_root=OUTPUT_ROOT,
            run_binding=RUN_BINDING,
        )
        write_json(OUTPUT_ROOT / "stage_status.json", status)
        print({"public_stage": stage_id, "action": "verified-complete"})

    STRICT_CAPACITY_RECEIPT_PATH = OUTPUT_ROOT / "strict_capacity_receipt.json"
    from vipibench.runtime_telemetry import strict_capacity_receipt_sha256

    if DEPENDENCY_PROFILE == "accelerator":
        ACCELERATOR_PATH = SESSION_RUNTIME_ROOT / "accelerator_capacity_check.json"
        ACCELERATOR_COMMAND = [
            sys.executable,
            "-m",
            "vipibench.cli",
            "check-accelerator",
            "--profile",
            str(PROJECT_ROOT / "configs" / "profiles" / "accelerator_80gb.yaml"),
            "--disk-path",
            str(OUTPUT_ROOT),
            "--output",
            str(ACCELERATOR_PATH),
        ]
        subprocess.check_call(ACCELERATOR_COMMAND, cwd=PROJECT_ROOT)
        accelerator = json.loads(ACCELERATOR_PATH.read_text(encoding="utf-8"))
        probe = accelerator.get("probe", {})
        if accelerator.get("status") != "PASS" or accelerator.get("hardware_observed") is not True:
            raise RuntimeError(accelerator.get("errors"))
        allowed_device_names = {
            "NVIDIA A100-SXM4-80GB",
            "NVIDIA A100-PCIE-80GB",
            "NVIDIA A100 80GB",
        }
        if (
            str(probe.get("device_name", "")) not in allowed_device_names
            or str(probe.get("compute_capability", "")) != "8.0"
            or float(probe.get("device_memory_gib", 0)) < 70
        ):
            raise RuntimeError("Observed device is not a registered NVIDIA A100 80GB target")
        if not STRICT_CAPACITY_RECEIPT_PATH.is_file():
            subprocess.check_call(
                CLI
                + [
                    "build-strict-capacity-receipt",
                    "--runtime-check",
                    str(ACCELERATOR_PATH),
                    "--project-root",
                    str(PROJECT_ROOT),
                    "--output",
                    str(STRICT_CAPACITY_RECEIPT_PATH),
                ],
                cwd=PROJECT_ROOT,
            )
        minimum_free_disk_gib = 80.0
    strict_capacity_receipt = json.loads(
        STRICT_CAPACITY_RECEIPT_PATH.read_text(encoding="utf-8")
    )
    strict_capacity_hash = strict_capacity_receipt_sha256(
        strict_capacity_receipt, project_root=PROJECT_ROOT
    )

    from vipibench.runtime_storage import verify_runtime_storage_plan

    RUNTIME_STORAGE_PLAN_PATH = Path(
        os.environ.get("VIPIBENCH_RUNTIME_STORAGE_PLAN_PATH", "")
    ).resolve()
    expected_storage_plan_sha256 = os.environ.get(
        "VIPIBENCH_RUNTIME_STORAGE_PLAN_SHA256", ""
    )
    expected_storage_plan_file_sha256 = os.environ.get(
        "VIPIBENCH_RUNTIME_STORAGE_PLAN_FILE_SHA256", ""
    )
    if not RUNTIME_STORAGE_PLAN_PATH.is_file():
        raise FileNotFoundError("Runtime storage plan is required")
    if sha256_file(RUNTIME_STORAGE_PLAN_PATH) != expected_storage_plan_file_sha256:
        raise ValueError("Runtime storage plan file hash mismatch")
    runtime_storage_plan = verify_runtime_storage_plan(
        json.loads(RUNTIME_STORAGE_PLAN_PATH.read_text(encoding="utf-8")),
        output_root=OUTPUT_ROOT,
        protected_roots=[PROJECT_ROOT],
    )
    if runtime_storage_plan["model_cache_root"] != str(EPHEMERAL_MODEL_CACHE_ROOT):
        raise ValueError("Runtime storage plan and model-cache binding disagree")
    if runtime_storage_plan["plan_sha256"] != expected_storage_plan_sha256:
        raise ValueError("Runtime storage plan payload hash mismatch")

    import shutil

    disk_roots = {
        "project": PROJECT_ROOT,
        "output": OUTPUT_ROOT,
        "model_cache": EPHEMERAL_MODEL_CACHE_ROOT,
        "session_temp": Path(str(runtime_storage_plan["session_temp_root"])),
        "ephemeral": Path(str(runtime_storage_plan["ephemeral_root"])),
    }
    disk_free_gib = {
        name: shutil.disk_usage(path).free / (1024**3) for name, path in disk_roots.items()
    }
    if any(value < minimum_free_disk_gib for value in disk_free_gib.values()):
        raise RuntimeError(
            {"disk_free_below_minimum": disk_free_gib, "minimum_gib": minimum_free_disk_gib}
        )

    LAUNCH_RECORD_PATH = OUTPUT_ROOT / "launch_record.json"
    if DEPENDENCY_PROFILE == "accelerator":
        write_json(
            LAUNCH_RECORD_PATH,
            {
                "schema_version": "1.0.0",
                "status": "PASS",
                "mode": MODE,
                "session_id": SESSION_ID,
                "selected_public_stage": SELECTED_STAGE,
                "dependency_profile": DEPENDENCY_PROFILE,
                "output_root": str(OUTPUT_ROOT),
                "durable_output_confirmed": True,
                "confirmatory_run_approved": True,
                "upload_authorized": True,
                "paid_compute_authorized": True,
                "technical_decision_owner": "training_pipeline",
                "autonomous_session_hard_ceiling_hours": autonomous_policy.hard_ceiling_hours,
                "autonomous_execution_policy": autonomous_policy.as_dict(),
                "autonomous_execution_policy_sha256": autonomous_policy_sha256,
                "launch_authorization_path": str(authorization_path),
                "launch_authorization_sha256": authorization_sha256,
                "observed_device_name": probe.get("device_name"),
                "resume_existing": resume_existing,
                "runtime_storage_plan_path": str(RUNTIME_STORAGE_PLAN_PATH),
                "runtime_storage_plan_sha256": expected_storage_plan_sha256,
                "runtime_storage_plan_file_sha256": expected_storage_plan_file_sha256,
                "ephemeral_root": runtime_storage_plan["ephemeral_root"],
                "ephemeral_model_cache_root": str(EPHEMERAL_MODEL_CACHE_ROOT),
                "session_temp_root": runtime_storage_plan["session_temp_root"],
                "scratch_same_device_as_output": runtime_storage_plan["same_device_as_output"],
                "cache_rotation_enabled": True,
                "disk_free_gib": disk_free_gib,
                "preflight_sha256": sha256_file(PREFLIGHT_PATH),
                "accelerator_sha256": sha256_file(ACCELERATOR_PATH),
                "strict_capacity_receipt_sha256": strict_capacity_hash,
                "launch_hashes": PREFLIGHT["launch_hashes"],
            },
        )
        complete_public_stage(
            "preflight",
            direct_outputs=[
                PREFLIGHT_PATH,
                ACCELERATOR_PATH,
                STRICT_CAPACITY_RECEIPT_PATH,
                RUNTIME_STORAGE_PLAN_PATH,
                LAUNCH_RECORD_PATH,
            ],
            always=True,
        )
    if SELECTED_STAGE not in {"preflight", "all"}:
        require_stage_prerequisite(
            plan=STAGE_PLAN,
            stage_id=SELECTED_STAGE,
            stage_ledger=ORCHESTRATION_LEDGER,
            group_ledger=STAGE_GROUP_LEDGER,
            output_root=OUTPUT_ROOT,
            run_binding=RUN_BINDING,
        )

In [ ]:
if MODE == "confirmatory":
    rebuild_root = OUTPUT_ROOT / "deterministic_rebuild"
    rebuilt_contrast = rebuild_root / "provenance_contrast.jsonl"
    rebuilt_manifest = rebuild_root / "provenance_contrast_manifest.json"
    rebuilt_audit = rebuild_root / "provenance_contrast_audit.json"
    run_checkpointed_stage(
        "compile-provenance-contrast",
        CLI
        + [
            "compile-provenance-contrast",
            "--config",
            str(PROJECT_ROOT / "configs/benchmark/provenance_contrast.yaml"),
            "--output",
            str(rebuilt_contrast),
            "--manifest",
            str(rebuilt_manifest),
        ],
        [rebuilt_contrast, rebuilt_manifest],
    )
    run_checkpointed_stage(
        "audit-provenance-contrast",
        CLI
        + [
            "audit-provenance-contrast",
            str(rebuilt_contrast),
            "--output",
            str(rebuilt_audit),
        ],
        [rebuilt_audit],
    )
    frozen_contrast = PROJECT_ROOT / "data/processed/provenance_contrast.jsonl"
    if stage_enabled(SELECTED_STAGE, "data") and sha256_file(
        rebuilt_contrast
    ) != sha256_file(frozen_contrast):
        raise RuntimeError(
            "Deterministic provenance-contrast rebuild does not match the frozen input"
        )
    complete_public_stage("data")

In [ ]:
if MODE == "confirmatory":
    import yaml

    session_config_dir = OUTPUT_ROOT / "configs"
    session_config_dir.mkdir(parents=True, exist_ok=True)
    tfidf_source = PROJECT_ROOT / "configs/models/tfidf_core.yaml"
    tfidf_config = yaml.safe_load(tfidf_source.read_text(encoding="utf-8"))
    tfidf_config.update(
        {
            "train_path": str(SPLIT_DIR / "train.jsonl"),
            "dev_path": str(SPLIT_DIR / "dev.jsonl"),
            "test_path": str(SPLIT_DIR / "test.jsonl"),
            "contrast_dataset": str(PROJECT_ROOT / "data/processed/provenance_contrast.jsonl"),
            "output_dir": str(OUTPUT_ROOT / "tfidf/model"),
        }
    )
    tfidf_session = session_config_dir / "tfidf_core.session.yaml"
    tfidf_session.write_text(yaml.safe_dump(tfidf_config, sort_keys=False), encoding="utf-8")
    tfidf_root = OUTPUT_ROOT / "tfidf"
    tfidf_outputs = [
        tfidf_root / "baseline_manifest.json",
        tfidf_root / "model/model.joblib",
        tfidf_root / "model/train_manifest.json",
        tfidf_root / "dev_predictions.jsonl",
        tfidf_root / "test_predictions.jsonl",
        tfidf_root / "thresholds.json",
        tfidf_root / "evaluation.json",
    ]
    run_checkpointed_stage(
        "tfidf-baseline",
        CLI
        + [
            "run-tfidf-baseline",
            "--project-root",
            str(PROJECT_ROOT),
            "--config",
            str(tfidf_session),
            "--output",
            str(tfidf_root / "baseline_manifest.json"),
        ],
        tfidf_outputs,
    )

    public_root = OUTPUT_ROOT / "public_detector"
    public_outputs = [
        public_root / "benchmark_manifest.json",
        public_root / "capacity_plan.json",
        public_root / "dev_predictions.jsonl",
        public_root / "test_predictions.jsonl",
        public_root / "thresholds.json",
        public_root / "evaluation.json",
    ]
    run_checkpointed_stage(
        "public-detector-benchmark",
        CLI
        + [
            "run-public-detector-benchmark",
            "--config",
            str(PROJECT_ROOT / "configs/models/public_detector.yaml"),
            "--split-dir",
            str(SPLIT_DIR),
            "--output-root",
            str(public_root),
        ],
        public_outputs,
    )
    complete_public_stage("baselines")

    encoder_root = OUTPUT_ROOT / "mdeberta"
    encoder_run_ids = [
        f"mdeberta-{mode}-s{seed}"
        for mode in ("role_only", "text_only", "text_role")
        for seed in (17, 29, 43)
    ]
    encoder_outputs = [encoder_root / "model_selection.json"]
    for run_id in encoder_run_ids:
        encoder_outputs.extend(
            [
                encoder_root / run_id / "training_decision.json",
                encoder_root / run_id / "test_prediction_manifest.json",
            ]
        )
    run_checkpointed_stage(
        "encoder-matrix",
        CLI
        + [
            "run-encoder-accelerator-matrix",
            "--config",
            str(PROJECT_ROOT / "configs/models/mdeberta_core.yaml"),
            "--split-dir",
            str(SPLIT_DIR),
            "--output-root",
            str(encoder_root),
        ],
        encoder_outputs,
    )
    ablation_path = OUTPUT_ROOT / "encoder_ablation_analysis.json"
    complete_public_stage("encoder")
    print(reset_ephemeral_model_cache(EPHEMERAL_MODEL_CACHE_ROOT))

In [ ]:
if MODE == "confirmatory":
    selection_path = OUTPUT_ROOT / "mdeberta/model_selection.json"
    selection_required_stages = {
        "encoder", "core", "attack-generate", "attack-evaluate",
        "analysis", "finalize", "all",
    }
    if not selection_path.is_file() and SELECTED_STAGE in selection_required_stages:
        raise FileNotFoundError("verified encoder model selection is required")
    if selection_path.is_file():
        selection = json.loads(selection_path.read_text(encoding="utf-8"))
        if selection.get("test_accessed") is not False:
            raise RuntimeError("Model selection must remain development-only")
        selected_run = selection["selected"]["run_id"]
        detector_model_version = selection["selected"]["model_artifact_version"]
        selected = OUTPUT_ROOT / "mdeberta" / selected_run
        print({"selected_run": selected_run, "selection_metric": selection["selection_metric"]})
    else:
        selected_run = None
        detector_model_version = None
        selected = OUTPUT_ROOT / "mdeberta" / "__not_selected__"

In [ ]:
if MODE == "confirmatory":
    core_trajectories = OUTPUT_ROOT / "core_target_trajectories.jsonl"
    run_checkpointed_stage(
        "core-target-trajectories",
        CLI
        + [
            "run-target-agent",
            "--dataset",
            str(SPLIT_DIR / "test.jsonl"),
            "--output",
            str(core_trajectories),
            "--checkpoint-dir",
            str(OUTPUT_ROOT / "core_target_checkpoints"),
            "--config",
            str(PROJECT_ROOT / "configs/models/target_agent.yaml"),
            "--strict-capacity-receipt",
            str(STRICT_CAPACITY_RECEIPT_PATH),
        ],
        [
            core_trajectories,
            core_trajectories.with_suffix(".run.json"),
            core_trajectories.with_suffix(".telemetry.json"),
        ],
    )
    print(reset_ephemeral_model_cache(EPHEMERAL_MODEL_CACHE_ROOT))
    four_arm_path = OUTPUT_ROOT / "static_four_arm_evaluation.json"
    run_checkpointed_stage(
        "static-four-arm-evaluation",
        CLI
        + [
            "evaluate-four-arms",
            "--predictions",
            str(selected / "core_test_predictions.jsonl"),
            "--trajectories",
            str(core_trajectories),
            "--thresholds",
            str(selected / "thresholds.json"),
            "--detector-model-version",
            detector_model_version,
            "--test-dataset",
            str(SPLIT_DIR / "test.jsonl"),
            "--output",
            str(four_arm_path),
        ],
        [four_arm_path],
    )
    complete_public_stage("core")

In [ ]:
if MODE == "confirmatory":
    candidate_dataset = OUTPUT_ROOT / "attack_candidates.jsonl"
    candidate_scores = OUTPUT_ROOT / "attack_candidate_scores.jsonl"
    run_checkpointed_stage(
        "attack-candidate-generation",
        CLI
        + [
            "generate-attack-candidates",
            "--project-root",
            str(PROJECT_ROOT),
            "--detector-model-dir",
            str(selected / "model"),
            "--output-dataset",
            str(candidate_dataset),
            "--output-scores",
            str(candidate_scores),
            "--checkpoint-dir",
            str(OUTPUT_ROOT / "attack_search_checkpoints"),
            "--config",
            str(PROJECT_ROOT / "configs/generation/adaptive_generator.yaml"),
        ],
        [
            candidate_dataset,
            candidate_scores,
            candidate_dataset.with_suffix(".validity.json"),
            candidate_dataset.with_suffix(".manifest.json"),
        ],
    )
    complete_public_stage("attack-generate")
    print(reset_ephemeral_model_cache(EPHEMERAL_MODEL_CACHE_ROOT))
    candidate_trajectories = OUTPUT_ROOT / "attack_target_trajectories.jsonl"
    run_checkpointed_stage(
        "attack-target-trajectories",
        CLI
        + [
            "run-target-agent",
            "--dataset",
            str(candidate_dataset),
            "--output",
            str(candidate_trajectories),
            "--checkpoint-dir",
            str(OUTPUT_ROOT / "attack_target_checkpoints"),
            "--config",
            str(PROJECT_ROOT / "configs/models/target_agent.yaml"),
            "--strict-capacity-receipt",
            str(STRICT_CAPACITY_RECEIPT_PATH),
        ],
        [
            candidate_trajectories,
            candidate_trajectories.with_suffix(".run.json"),
            candidate_trajectories.with_suffix(".telemetry.json"),
        ],
    )
    print(reset_ephemeral_model_cache(EPHEMERAL_MODEL_CACHE_ROOT))
    RUNTIME_TELEMETRY_PATH = OUTPUT_ROOT / "runtime_telemetry.json"
    run_checkpointed_stage(
        "consolidate-runtime-telemetry",
        CLI
        + [
            "consolidate-runtime-telemetry",
            "--telemetry",
            str(core_trajectories.with_suffix(".telemetry.json")),
            "--telemetry",
            str(candidate_trajectories.with_suffix(".telemetry.json")),
            "--strict-capacity-receipt",
            str(STRICT_CAPACITY_RECEIPT_PATH),
            "--project-root",
            str(PROJECT_ROOT),
            "--output",
            str(RUNTIME_TELEMETRY_PATH),
        ],
        [RUNTIME_TELEMETRY_PATH],
    )
    attack_evaluation = OUTPUT_ROOT / "attack_search_evaluation.json"
    run_checkpointed_stage(
        "attack-search-evaluation",
        CLI
        + [
            "evaluate-attack-search",
            "--candidate-dataset",
            str(candidate_dataset),
            "--candidate-scores",
            str(candidate_scores),
            "--target-trajectories",
            str(candidate_trajectories),
            "--thresholds",
            str(selected / "thresholds.json"),
            "--output",
            str(attack_evaluation),
        ],
        [attack_evaluation],
    )
    complete_public_stage("attack-evaluate")

In [ ]:
if MODE == "confirmatory" and SELECTED_STAGE in {"analysis", "finalize", "all"}:
    training_decision_paths = sorted((OUTPUT_ROOT / "mdeberta").glob("*/training_decision.json"))
    if len(training_decision_paths) != 9:
        raise FileNotFoundError(
            {"training_decision_count": len(training_decision_paths), "required": 9}
        )
    training_decisions = [
        json.loads(path.read_text(encoding="utf-8")) for path in training_decision_paths
    ]
    if any(
        item.get("decision_split") != "dev"
        or item.get("test_accessed") is not False
        or item.get("final_holdout_feedback_allowed") is not False
        for item in training_decisions
    ):
        raise RuntimeError("training decision artifact violated the development-only stop contract")
    capacity_plan_path = OUTPUT_ROOT / "mdeberta/capacity_plan.json"
    capacity_plan = json.loads(capacity_plan_path.read_text(encoding="utf-8"))
    capacity_numerics_canary = capacity_plan.get("numerics_canary", {})
    if (
        capacity_plan.get("status") != "PASS"
        or not isinstance(capacity_plan.get("selected"), dict)
        or capacity_numerics_canary.get("status") != "PASS"
        or capacity_numerics_canary.get("candidate_id")
        != capacity_plan["selected"].get("candidate_id")
    ):
        raise RuntimeError("capacity selection lacks a passing two-step numerical canary")
    encoder_analysis_outputs = [
        OUTPUT_ROOT / "mdeberta" / run_id / "test_manifest.json"
        for run_id in encoder_run_ids
    ]
    run_checkpointed_stage(
        "encoder-test-analysis",
        CLI
        + [
            "run-encoder-test-analysis",
            "--config",
            str(PROJECT_ROOT / "configs/models/mdeberta_core.yaml"),
            "--split-dir",
            str(SPLIT_DIR),
            "--output-root",
            str(OUTPUT_ROOT / "mdeberta"),
        ],
        encoder_analysis_outputs,
    )
    run_checkpointed_stage(
        "encoder-ablation-analysis",
        CLI
        + [
            "analyze-encoder-ablations",
            "--output-root",
            str(OUTPUT_ROOT / "mdeberta"),
            "--output",
            str(ablation_path),
        ],
        [ablation_path],
    )
    AUTONOMOUS_DECISIONS_PATH = OUTPUT_ROOT / "autonomous_decisions.json"
    autonomous_decisions = {
            "schema_version": "1.0.0",
            "status": "PASS",
            "technical_decision_owner": "training_pipeline",
            "execution_policy_sha256": autonomous_policy_sha256,
            "session_hard_ceiling_hours": autonomous_policy.hard_ceiling_hours,
            "capacity_plan_sha256": sha256_file(capacity_plan_path),
            "selected_capacity_candidate": capacity_plan["selected"],
            "capacity_numerics_canary": capacity_numerics_canary,
            "training_decision_hashes": {
                path.parent.name: sha256_file(path) for path in training_decision_paths
            },
            "selected_run": selected_run,
            "model_selection_sha256": sha256_file(OUTPUT_ROOT / "mdeberta/model_selection.json"),
            "normal_stop_signal": "dev:dev_auprc",
            "final_holdout_feedback_allowed": False,
            "resume_mode": autonomous_policy.resume_mode,
    }
    if stage_enabled(SELECTED_STAGE, "analysis"):
        write_json(AUTONOMOUS_DECISIONS_PATH, autonomous_decisions)
    SUPPORTING_EVIDENCE_PATH = OUTPUT_ROOT / "postrun_supporting_evidence.json"
    run_checkpointed_stage(
        "prepare-postrun-supporting-evidence",
        CLI
        + [
            "prepare-postrun-supporting-evidence",
            "--project-root",
            str(PROJECT_ROOT),
            "--output-root",
            str(OUTPUT_ROOT),
            "--launch-authorization-source",
            str(authorization_path),
            "--strict-capacity-receipt",
            str(STRICT_CAPACITY_RECEIPT_PATH),
            "--output",
            str(SUPPORTING_EVIDENCE_PATH),
        ],
        [
            SUPPORTING_EVIDENCE_PATH,
            OUTPUT_ROOT / "launch_authorization.json",
            OUTPUT_ROOT / "mdeberta/matrix_manifest.json",
            OUTPUT_ROOT / "final_holdout_format_fallback_ledger.json",
            OUTPUT_ROOT / "failure_ledger.json",
        ],
    )
    STATIC_ANALYSIS_PATH = OUTPUT_ROOT / "static_analysis.json"
    run_checkpointed_stage(
        "analyze-static-system",
        CLI
        + [
            "analyze-static-system",
            "--four-arm-report",
            str(OUTPUT_ROOT / "static_four_arm_evaluation.json"),
            "--trajectories",
            str(OUTPUT_ROOT / "core_target_trajectories.jsonl"),
            "--telemetry",
            str(RUNTIME_TELEMETRY_PATH),
            "--strict-capacity-receipt",
            str(STRICT_CAPACITY_RECEIPT_PATH),
            "--output",
            str(STATIC_ANALYSIS_PATH),
        ],
        [STATIC_ANALYSIS_PATH],
    )
    RQ2_ANALYSIS_PATH = OUTPUT_ROOT / "rq2_analysis.json"
    run_checkpointed_stage(
        "analyze-rq2-diagnostics",
        CLI
        + [
            "analyze-rq2-diagnostics",
            "--output-root",
            str(OUTPUT_ROOT / "mdeberta"),
            "--control-identity",
            str(ablation_path),
            "--analysis-config",
            str(PROJECT_ROOT / "configs/experiments/confirmatory_analysis.yaml"),
            "--output",
            str(RQ2_ANALYSIS_PATH),
        ],
        [RQ2_ANALYSIS_PATH],
    )
    H3_ANALYSIS_PATH = OUTPUT_ROOT / "h3_analysis.json"
    run_checkpointed_stage(
        "analyze-h3",
        CLI
        + [
            "analyze-h3",
            "--four-arm-report",
            str(OUTPUT_ROOT / "static_four_arm_evaluation.json"),
            "--static-analysis",
            str(STATIC_ANALYSIS_PATH),
            "--analysis-config",
            str(PROJECT_ROOT / "configs/experiments/confirmatory_analysis.yaml"),
            "--output",
            str(H3_ANALYSIS_PATH),
        ],
        [H3_ANALYSIS_PATH],
    )
    ADAPTIVE_ANALYSIS_PATH = OUTPUT_ROOT / "adaptive_analysis.json"
    run_checkpointed_stage(
        "analyze-adaptive-search",
        CLI
        + [
            "analyze-adaptive-search",
            "--candidate-dataset",
            str(OUTPUT_ROOT / "attack_candidates.jsonl"),
            "--candidate-validity",
            str(OUTPUT_ROOT / "attack_candidates.validity.json"),
            "--candidate-manifest",
            str(OUTPUT_ROOT / "attack_candidates.manifest.json"),
            "--adaptive-report",
            str(OUTPUT_ROOT / "attack_search_evaluation.json"),
            "--generator-config",
            str(PROJECT_ROOT / "configs/generation/adaptive_generator.yaml"),
            "--telemetry",
            str(RUNTIME_TELEMETRY_PATH),
            "--strict-capacity-receipt",
            str(STRICT_CAPACITY_RECEIPT_PATH),
            "--output",
            str(ADAPTIVE_ANALYSIS_PATH),
        ],
        [ADAPTIVE_ANALYSIS_PATH],
    )
    complete_public_stage(
        "analysis", direct_outputs=[AUTONOMOUS_DECISIONS_PATH]
    )
    PRE_AUDIT_CONTEXT_PATH = OUTPUT_ROOT / "run_manifest.pre_audit.json"
    run_checkpointed_stage(
        "write-postrun-run-context",
        CLI
        + [
            "write-postrun-run-context",
            "--project-root",
            str(PROJECT_ROOT),
            "--output-root",
            str(OUTPUT_ROOT),
            "--output",
            str(PRE_AUDIT_CONTEXT_PATH),
        ],
        [PRE_AUDIT_CONTEXT_PATH],
    )
    RAW_MANIFEST_STAGE_PATH = OUTPUT_ROOT / "postrun_raw_manifests_stage.json"
    run_checkpointed_stage(
        "build-postrun-raw-manifests",
        CLI
        + [
            "build-postrun-raw-manifests",
            "--project-root",
            str(PROJECT_ROOT),
            "--output-root",
            str(OUTPUT_ROOT),
            "--output",
            str(RAW_MANIFEST_STAGE_PATH),
        ],
        [
            RAW_MANIFEST_STAGE_PATH,
            OUTPUT_ROOT / "raw_predictions_manifest.json",
            OUTPUT_ROOT / "raw_trajectories_manifest.json",
        ],
    )
    POSTRUN_AUDIT_PATH = OUTPUT_ROOT / "postrun_audit.json"
    run_checkpointed_stage(
        "audit-postrun",
        CLI
        + [
            "audit-postrun",
            "--project-root",
            str(PROJECT_ROOT),
            "--output-root",
            str(OUTPUT_ROOT),
            "--output",
            str(POSTRUN_AUDIT_PATH),
        ],
        [POSTRUN_AUDIT_PATH],
    )
    RUN_MANIFEST_PATH = OUTPUT_ROOT / "run_manifest.json"
    run_checkpointed_stage(
        "finalize-confirmatory-run",
        CLI
        + [
            "finalize-confirmatory-run",
            "--project-root",
            str(PROJECT_ROOT),
            "--output-root",
            str(OUTPUT_ROOT),
            "--postrun-audit",
            str(POSTRUN_AUDIT_PATH),
            "--output",
            str(RUN_MANIFEST_PATH),
        ],
        [RUN_MANIFEST_PATH],
    )
    REPORT_PACKAGE_ROOT = OUTPUT_ROOT / "bao_cao_hinh_anh"
    REPORT_ASSET_MANIFEST_PATH = REPORT_PACKAGE_ROOT / "report_assets_manifest.json"
    REPORT_ARCHIVE_PATH = REPORT_PACKAGE_ROOT / "goi_hinh_bao_cao.zip"
    run_checkpointed_stage(
        "materialize-report-assets",
        CLI
        + [
            "materialize-report-assets",
            "--project-root",
            str(PROJECT_ROOT),
            "--output-root",
            str(OUTPUT_ROOT),
        ],
        [REPORT_ASSET_MANIFEST_PATH, REPORT_ARCHIVE_PATH],
    )
    complete_public_stage("finalize")
    if stage_enabled(SELECTED_STAGE, "finalize"):
        final_run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding="utf-8"))
        print(
            {
                "RUN_COMPLETE": final_run_manifest["RUN_COMPLETE"],
                "RESEARCH_EVIDENCE_ELIGIBLE": final_run_manifest["RESEARCH_EVIDENCE_ELIGIBLE"],
                "run_manifest": str(RUN_MANIFEST_PATH),
                "goi_hinh_bao_cao": str(REPORT_ARCHIVE_PATH),
            }
        )